In [90]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import Trainer
import numpy as np

class LeaveOneOutTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwarg):
        """
        Custom loss function with leave-one-out adaptation.
        
        Steps:
        1. Standard cross-entropy loss on in-domain topics.
        2. Leave-one-out strategy: Pick one topic to remove.
        3. Evaluate adaptation performance on the left-out topic.
        4. Adjust loss based on how well it generalizes.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Standard classification loss
        loss_ce = F.cross_entropy(logits, labels)
        
        # --- Leave-One-Out Adaptation ---
        unique_topics = torch.unique(labels)
        if len(unique_topics) > 1:
            leave_out_topic = unique_topics[torch.randint(len(unique_topics), (1,))].item()
            
            # Get samples *not* from the leave-out topic
            mask_in_domain = labels != leave_out_topic
            logits_in_domain = logits[mask_in_domain]
            labels_in_domain = labels[mask_in_domain]
            
            # Get samples *from* the leave-out topic
            mask_out_domain = labels == leave_out_topic
            logits_out_domain = logits[mask_out_domain]
            labels_out_domain = labels[mask_out_domain]
            
            if logits_out_domain.size(0) > 0:
                # Compute loss on the left-out topic
                loss_leave_out = F.cross_entropy(logits_out_domain, labels_out_domain)
                
                # Adaptation penalty: encourage generalization
                loss = loss_ce + 0.5 * loss_leave_out
            else:
                loss = loss_ce  # No adaptation needed if no left-out examples
        else:
            loss = loss_ce  # Only one topic, normal training
        
        return (loss, outputs) if return_outputs else loss


In [69]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load the model
model_path = "./loo_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)



In [16]:
from transformers import BertTokenizerFast
import pandas as pd
import numpy as np
from datasets import Dataset

# Load your dataset
df_train = pd.read_csv("df_train_sbert.csv")  # Adjust this as needed

# Load topic definitions
df_topics = pd.read_csv("topics_definitions.csv")  # Ensure it has columns ["topic_id", "topic_definition"]

# choose a pseudo new topic, leave it completly for testing
pseudo_new_topic_id = np.random.choice(df_train['topic_id'].unique())
df_train = df_train[df_train['topic_id'] != pseudo_new_topic_id]
#df_topics = df_topics[df_topics['topic_id'] != pseudo_new_topic_id]

# only choose the data with topic_id for training # regardless of positive or negative label
df_train = df_train.dropna(subset = ['topic_id'])

# Map topic_id to topic description
topic_dict = df_topics.set_index("topic_id")["topic_definition"].to_dict()

# Merge topic descriptions into df_train
df_train["topic_definition"] = df_train["topic_id"].map(topic_dict)


# Tokenization for BERT
#tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        examples["topic_definition"],  # Pair with topic description
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_train)
train_dataset = train_dataset.map(tokenize_function, batched=True)

# Format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/41894 [00:00<?, ? examples/s]

In [71]:
# prepare test data set
# for test, we keep all topics, include the pseudo new 
df_test = pd.read_csv('df_test_sbert.csv')
df_test = df_test.dropna(subset = ['topic_id'])
df_test["topic_definition"] = df_test["topic_id"].map(topic_dict)

# Convert to Hugging Face Dataset
test_dataset = Dataset.from_pandas(df_test)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format for PyTorch
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/11786 [00:00<?, ? examples/s]

In [82]:
import torch
import torch.nn.functional as F
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Define compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # Convert logits to predicted labels
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

#train_dataset_small = train_dataset.select(range(100))  # Take the first 100 samples


# Define Training Arguments

training_args = TrainingArguments(
    output_dir="./results_leave_one_out",
    num_train_epochs=3,  # Training for 3 epochs
    per_device_train_batch_size=16,  # Training batch size of 16
    per_device_eval_batch_size=32,  # Evaluation batch size of 32
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Save the model at the end of each epoch
    logging_dir="./logs_leave_one_out",  # Logging directory
    logging_steps=10,  # Log every 10 steps
    logging_strategy="epoch",  # Log at the end of each epoch
    load_best_model_at_end=True,  # Load the best model based on validation metric
    metric_for_best_model="accuracy",  # Select best model based on accuracy
)




# Initialize the BERT model
num_labels = 2  # Binary classification (in-topic vs out-of-topic)
#model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
#model_name = "distilbert-base-uncased"
#model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


# Initialize Trainer with Leave-One-Out loss
trainer = LeaveOneOutTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # Prepared dataset
    eval_dataset=test_dataset,    # Evaluates on pseudo-new topic
    compute_metrics=compute_metrics,
)

# Train Model
#trainer.train()


In [34]:
# Save model
trainer.save_model("./loo_model")

# Save tokenizer (important!)
tokenizer.save_pretrained("./loo_model")

# Save training arguments
torch.save(training_args, "./loo_model/training_args.bin")


In [75]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Get predictions on the test dataset
outputs = trainer.predict(test_dataset)

# Extract logits and true labels
logits = torch.tensor(outputs.predictions)  # Model outputs logits
labels = torch.tensor(outputs.label_ids)  # True labels


In [76]:
# Apply softmax to convert logits to probabilities
probs = torch.softmax(logits, dim=-1)

# Convert probabilities to class predictions (threshold at 0.5)
predictions = torch.argmax(probs, dim=-1)  # Take the index with max probability


In [77]:
import numpy as np

# Ensure labels and predictions are NumPy arrays
labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.array(labels)
predictions_np = predictions.cpu().numpy() if isinstance(predictions, torch.Tensor) else np.array(predictions)

# Compute metrics
accuracy = accuracy_score(labels_np, predictions_np)
precision, recall, f1, _ = precision_recall_fscore_support(labels_np, predictions_np, average="binary", zero_division=1)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")


Accuracy: 0.9171
Precision: 0.8847
Recall: 0.8638
F1-score: 0.8741


In [78]:
import numpy as np

# Ensure labels are in NumPy format
labels_np = np.array(labels)

# Use Dataset.map() to add predictions
test_dataset = test_dataset.map(lambda example, idx: {"predicted_label": labels_np[idx]}, with_indices=True)


Map:   0%|          | 0/11786 [00:00<?, ? examples/s]

In [43]:
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

topic_ids = test_dataset["topic_id"]  # Extract topic IDs

# Step 3: Compute Metrics per `topic_id`
topic_metrics = defaultdict(list)

for topic_id, true_label, pred_label in zip(topic_ids, labels.tolist(), predictions.tolist()):
    topic_metrics[topic_id].append((true_label, pred_label))

# Step 4: Calculate Metrics for Each `topic_id`
results = []
for topic_id, values in topic_metrics.items():
    y_true = [x[0] for x in values]
    y_pred = [x[1] for x in values]

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=1)

    results.append({
        "topic_id": topic_id,
        "accuracy": round(acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4)
    })

# Step 5: Convert to DataFrame and Print
df_results = pd.DataFrame(results)
print(df_results)


   topic_id  accuracy  precision  recall  f1_score
0     552.0    0.9724     0.7941  0.7297    0.7606
1     543.0    0.7124     0.8742  0.4863    0.6250
2     547.0    0.9599     0.9388  0.9517    0.9452
3     550.0    0.9041     0.8960  0.9050    0.9005
4     556.0    0.9792     0.8197  0.7812    0.8000
5     602.0    0.8777     0.8489  0.9595    0.9008
6     546.0    0.9189     0.8810  0.9564    0.9172
7     544.0    0.9420     0.9312  0.9479    0.9395
8     600.0    0.9487     0.8611  0.9435    0.9004
9     554.0    0.9606     0.8182  0.7692    0.7930


In [23]:
print('the pseudo new topic id is:', pseudo_new_topic_id)

the pseudo new topic id is: 543.0


In [91]:
import torch
import numpy as np
import pandas as pd
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
from datasets import Dataset


# Define fine-tuning arguments
fine_tune_args = TrainingArguments(
    output_dir="./results_few_shot",
    num_train_epochs=3,  # Few epochs for refinement
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_few_shot",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=5e-6,  # Lower LR to prevent overfitting
    warmup_ratio=0.06,
    weight_decay=0.1,
)

# Select top-K confidently predicted quotes
K_values = [32, 64]  # Trying both K=32 and K=64
few_shot_results = {}

for K in K_values:

    model_path = "./loo_model"
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels = 2)
    model.config.problem_type = "single_label_classification"
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    print(f"\n🔹 Fine-tuning with K={K} samples")

    # Select K quotes for fine-tuning
    top_k_subset = test_dataset.filter(lambda example: 
        example["topic_id"] == pseudo_new_topic_id and example["predicted_label"] == 1
    ).select(range(min(K, len(test_dataset))))  # Ensure K does not exceed dataset size
    
    top_k_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    # Get indices of top K samples
    top_k_indices = set(top_k_subset["review_id"])  # Assuming each sample has a unique "id"

    # Filter test set to exclude these samples
    filtered_test_dataset = test_dataset.filter(lambda example: example["review_id"] not in top_k_indices)
    
    top_k_subset = top_k_subset.map(lambda x: {"label": int(x["label"])})
    filtered_test_dataset = filtered_test_dataset.map(lambda x: {"label": int(x["label"])})
    # ✅ Ensure label is `torch.long` (int64)
    top_k_subset = top_k_subset.map(lambda x: {"label": torch.tensor(x["label"], dtype=torch.long)})
    filtered_test_dataset = filtered_test_dataset.map(lambda x: {"label": torch.tensor(x["label"], dtype=torch.long)})
    top_k_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    filtered_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    


    # Fine-tune the model
    trainer_refine = Trainer(
        model=model,
        args=fine_tune_args,
        train_dataset=top_k_subset,  # Now properly formatted
        eval_dataset=filtered_test_dataset,  # Evaluate on full test set
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer_refine.train()

    # Evaluate the fine-tuned model
    eval_results = trainer_refine.evaluate()
    few_shot_results[K] = eval_results
    print(f"\n🔹 Fine-tuning results for K={K}: {eval_results}")

# Compare results for K=32 and K=64
print("\nFinal comparison:")
for K, results in few_shot_results.items():
    print(f"K={K}: Accuracy={results['eval_accuracy']:.4f}, Loss={results['eval_loss']:.4f}")



🔹 Fine-tuning with K=32 samples


Filter:   0%|          | 0/11786 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11786 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/11753 [00:00<?, ? examples/s]

...

/var/folders/bj/w3wvgp6n4118_ksndj577lb00000gn/T/ipykernel_1905/7750475.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  top_k_subset = top_k_subset.map(lambda x: {"label": torch.tensor(x["label"], dtype=torch.long)})


Map:   0%|          | 0/11720 [00:00<?, ? examples/s]

/var/folders/bj/w3wvgp6n4118_ksndj577lb00000gn/T/ipykernel_1905/7750475.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  filtered_test_dataset = filtered_test_dataset.map(lambda x: {"label": torch.tensor(x["label"], dtype=torch.long)})
/var/folders/bj/w3wvgp6n4118_ksndj577lb00000gn/T/ipykernel_1905/7750475.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_refine = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.277860,0.925171,0.876989,0.899016,0.887866
2,No log,0.267097,0.928242,0.868864,0.921284,0.894307
3,1.356000,0.265634,0.928840,0.866054,0.927499,0.895724



🔹 Fine-tuning results for K=64: {'eval_loss': 0.2656344473361969, 'eval_accuracy': 0.928839590443686, 'eval_precision': 0.8660541586073501, 'eval_recall': 0.9274987053340238, 'eval_f1': 0.8957239309827457, 'eval_runtime': 94.8156, 'eval_samples_per_second': 123.608, 'eval_steps_per_second': 3.871, 'epoch': 3.0}

Final comparison:
K=32: Accuracy=0.9256, Loss=0.2754
K=64: Accuracy=0.9288, Loss=0.2656


In [95]:
# Save model
trainer_refine.save_model("./loo_model_finetune_64")

# Save tokenizer (important!)
tokenizer.save_pretrained("./loo_model_finetune_64")

# Save training arguments
torch.save(training_args, "./loo_model_finetune_64/training_args.bin")

In [92]:
import torch

# Get predictions on the test dataset
outputs = trainer_refine.predict(filtered_test_dataset)

# Extract logits and true labels
logits = torch.tensor(outputs.predictions)  # Model outputs logits
labels = torch.tensor(outputs.label_ids)  # True labels

# Apply softmax to convert logits to probabilities
probs = torch.softmax(logits, dim=-1)

# Convert probabilities to class predictions (threshold at 0.5)
predictions = torch.argmax(probs, dim=-1)  # Take the index with max probability

In [94]:
import numpy as np

# Ensure labels and predictions are NumPy arrays
labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.array(labels)
predictions_np = predictions.cpu().numpy() if isinstance(predictions, torch.Tensor) else np.array(predictions)

# Compute metrics
accuracy = accuracy_score(labels_np, predictions_np)
precision, recall, f1, _ = precision_recall_fscore_support(labels_np, predictions_np, average="binary", zero_division=1)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Accuracy: 0.9288
Precision: 0.8661
Recall: 0.9275
F1-score: 0.8957


In [93]:
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

topic_ids = filtered_test_dataset["topic_id"]  # Extract topic IDs

# Step 3: Compute Metrics per `topic_id`
topic_metrics = defaultdict(list)

for topic_id, true_label, pred_label in zip(topic_ids, labels.tolist(), predictions.tolist()):
    topic_metrics[topic_id].append((true_label, pred_label))

# Step 4: Calculate Metrics for Each `topic_id`
results = []
for topic_id, values in topic_metrics.items():
    y_true = [x[0] for x in values]
    y_pred = [x[1] for x in values]

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=1)

    results.append({
        "topic_id": topic_id,
        "accuracy": round(acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4)
    })

# Step 5: Convert to DataFrame and Print
df_results = pd.DataFrame(results)
print(df_results)

   topic_id  accuracy  precision  recall  f1_score
0     552.0    0.9756     0.7750  0.8378    0.8052
1     543.0    0.8542     0.8623  0.8161    0.8386
2     547.0    0.9558     0.9263  0.9540    0.9400
3     550.0    0.9040     0.8727  0.9367    0.9035
4     556.0    0.9759     0.7465  0.8281    0.7852
5     602.0    0.8710     0.8369  0.9653    0.8965
6     546.0    0.9173     0.8699  0.9686    0.9166
7     544.0    0.9386     0.9145  0.9605    0.9370
8     600.0    0.9368     0.8244  0.9432    0.8798
9     554.0    0.9564     0.7778  0.7778    0.7778


result of loo_training:
topic_id  accuracy  precision  recall  f1_score
0     552.0    0.9724     0.7941  0.7297    0.7606
1     543.0    0.7124     0.8742  0.4863    0.6250
2     547.0    0.9599     0.9388  0.9517    0.9452
3     550.0    0.9041     0.8960  0.9050    0.9005
4     556.0    0.9792     0.8197  0.7812    0.8000
5     602.0    0.8777     0.8489  0.9595    0.9008
6     546.0    0.9189     0.8810  0.9564    0.9172
7     544.0    0.9420     0.9312  0.9479    0.9395
8     600.0    0.9487     0.8611  0.9435    0.9004
9     554.0    0.9606     0.8182  0.7692    0.7930

Accuracy: 0.9171
Precision: 0.8847
Recall: 0.8638
F1-score: 0.8741

result of 64-few_shot fine tune based on the loo model:
topic_id  accuracy  precision  recall  f1_score
0     552.0    0.9756     0.7750  0.8378    0.8052
1     543.0    0.8542     0.8623  0.8161    0.8386
2     547.0    0.9558     0.9263  0.9540    0.9400
3     550.0    0.9040     0.8727  0.9367    0.9035
4     556.0    0.9759     0.7465  0.8281    0.7852
5     602.0    0.8710     0.8369  0.9653    0.8965
6     546.0    0.9173     0.8699  0.9686    0.9166
7     544.0    0.9386     0.9145  0.9605    0.9370
8     600.0    0.9368     0.8244  0.9432    0.8798
9     554.0    0.9564     0.7778  0.7778    0.7778

Accuracy: 0.9288
Precision: 0.8661
Recall: 0.9275
F1-score: 0.8957

## 📌 Proposal: Few-Shot Learning for Topic Adaptation

### **🔍 Overview**
Our current hierarchical classification model demonstrates strong performance on **seen topics**, but struggles with **truly new topics** (e.g., Topic 544). The low recall for unseen topics suggests that the model is overly conservative in assigning new labels, leading to underrepresentation of these topics. 

To improve the model’s adaptability, we propose integrating **Few-Shot Learning (FSL)** into our framework. This will allow the model to generalize better to novel topics using minimal labeled data.

---

## **1️⃣ Current Observations**
### **✅ Strengths:**
- The model achieves **90%+ accuracy** on known topics.
- The hierarchical framework effectively separates **binary filtering** and **multiclass classification**.

### **⚠ Weaknesses:**
- **Significant performance drop** on truly new topics.
- **High precision, low recall** → Model is too conservative in assigning new topic labels.
- **Does not exploit topic descriptions or embeddings effectively**.

---

## **2️⃣ Few-Shot Learning Strategy**
We propose two approaches for few-shot adaptation:

### **🔹 Approach 1: Fine-Tune on Few Examples of the New Topic**
- Introduce **K-labeled examples** for the new topic (e.g., **K=5,10,20**).
- Fine-tune the model **incrementally** on these new samples.
- Assess **accuracy vs. data efficiency trade-off**.

**Advantages:**
- Minimal changes to the current framework.
- Can be integrated into the existing training pipeline.
- Requires only a small number of labeled quotes for new topics.

**Implementation Steps:**
1. Randomly sample **K quotes** from the pseudo-new topic (if available) or generate synthetic samples.
2. Append these to the training set.
3. Fine-tune the existing model while preserving knowledge on old topics.
4. Evaluate the improvement on **Topic 544 and unseen topics**.

---

### **🔹 Approach 2: Prototypical Networks for Topic Adaptation**
Instead of classifying quotes **directly**, we redefine the problem as:

> **“How similar is this quote to the topic description?”**

- Train the model to **embed quotes and topics into the same space**.
- Use a **distance-based classifier** (e.g., nearest neighbors in embedding space) instead of a fixed label classifier.
- New topics can be **added dynamically** without retraining.

**Advantages:**
- No need to retrain for every new topic.
- Leverages **topic embeddings** effectively.
- Better generalization to unseen categories.

**Implementation Steps:**
1. Convert each **topic description into an embedding** using SBERT/BERT.
2. Embed each **quote** in the same vector space.
3. Train the model using **contrastive loss** to pull similar topic-quote pairs closer together.
4. For inference, assign a quote to the **closest topic embedding**.

---

## **3️⃣ Evaluation Plan**
We will compare the following methods:
- **Baseline:** Current hierarchical classification (BERT)
- **Few-Shot Fine-Tuning (Approach 1)**
- **Prototypical Networks (Approach 2)**

Metrics for comparison:
- **Accuracy & F1-score on pseudo-new topics.**
- **Data efficiency:** How many new examples are needed to reach optimal performance?
- **Generalization to unseen topics.**

---

## **4️⃣ Next Steps**
1. **Prepare few-shot training data** (K-samples for pseudo-new topic 544).
2. **Implement and fine-tune both approaches**.
3. **Evaluate performance on unseen topics.**
4. **Compare results & optimize strategy for real deployment.**

---

## **📌 Conclusion**
By integrating **Few-Shot Learning**, we aim to make our hierarchical classification model more robust to **newly introduced topics** while maintaining its strong performance on known topics. This will significantly reduce the need for full retraining and allow for **real-time topic adaptation** in production.

🚀 **Let’s experiment and refine the best approach!**
